# nuscenes.extractorsデバッグ用可視化スクリプト

In [ ]:
# extractor実施
import sys
from pathlib import Path

# matplotlib 3.5.x と matplotlib-inline 0.2.x の互換性対策
import matplotlib
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.carla_utils.carla_client import get_carla_client, set_world
from src.map_extraction.nuscenes.create_nusc_map import extract_carla_map_data

HOST = 'localhost'
PORT = 2000
MAP_NAME = None
SAMPLING_RESOLUTION = 1.0

# CARLA接続
client = get_carla_client(host=HOST, port=PORT)
world = set_world(client, map_name=MAP_NAME)
map_name = world.get_map().name.split('/')[-1]

# Extractor実行
carla_map_data = extract_carla_map_data(world, sampling_resolution=SAMPLING_RESOLUTION)

In [ ]:
# lanesのポリゴンをCOLOR_KEYSで指定したフィールドの組み合わせで色分けしてプロット
import matplotlib.pyplot as plt
import numpy as np

COLOR_KEYS = ["road_id"]  # List of "lane_id", "road_id", "section_id"

lanes = carla_map_data['lanes']

# COLOR_KEYSの組み合わせでユニークなグループを作成
groups = sorted(set(
    tuple(getattr(info, k) for k in COLOR_KEYS)
    for info in lanes.values()
))
cmap = plt.cm.get_cmap('tab20', len(groups))
group_to_color = {g: cmap(i) for i, g in enumerate(groups)}

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

for key, lane_info in lanes.items():
    if not lane_info.boundary_points:
        continue
    # 左境界 + 右境界(逆順) でポリゴンを構成
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right

    if len(polygon_coords) < 3:
        continue

    group = tuple(getattr(lane_info, k) for k in COLOR_KEYS)
    color = group_to_color[group]
    poly = plt.Polygon(polygon_coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.3, alpha=0.6)
    ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Lanes colored by {COLOR_KEYS} ({len(lanes)} lanes, {len(groups)} groups)')
plt.tight_layout()
plt.show()

In [ ]:
# lanesのポリゴンをlane_idの正負で色分けしてプロット
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

for key, lane_info in lanes.items():
    if not lane_info.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right

    if len(polygon_coords) < 3:
        continue

    if lane_info.lane_id > 0:
        color = 'skyblue'
    elif lane_info.lane_id < 0:
        color = 'salmon'
    else:
        color = 'gray'

    poly = plt.Polygon(polygon_coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.3, alpha=0.6)
    ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
n_pos = sum(1 for l in lanes.values() if l.lane_id > 0)
n_neg = sum(1 for l in lanes.values() if l.lane_id < 0)
ax.set_title(f'Lanes by lane_id sign (positive={n_pos} blue, negative={n_neg} red)')
plt.tight_layout()
plt.show()

In [ ]:
# Road Dividerの可視化
import matplotlib.pyplot as plt

# Junctionに属さないRoad Dividerのみを抽出
road_dividers = [d for d in carla_map_data['road_dividers']]

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてレーンポリゴンを描画
for lane_info in lanes.values():
    if not lane_info.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right
    if len(polygon_coords) >= 3:
        poly = plt.Polygon(polygon_coords, closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
        ax.add_patch(poly)

# Road Dividerを描画
for rd in road_dividers:
    if len(rd.points) < 2:
        continue
    xs = [p[0] for p in rd.points]
    ys = [p[1] for p in rd.points]
    ax.plot(xs, ys, color='red', linewidth=1.5, alpha=0.8)

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Road Dividers ({len(road_dividers)} lines)')
plt.tight_layout()
plt.show()

In [ ]:
# Lane Dividerの可視化
import matplotlib.pyplot as plt

# Junctionに属さないLane Dividerのみを抽出
lane_dividers = [d for d in carla_map_data['lane_dividers']]

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてレーンポリゴンを描画
for lane_info in lanes.values():
    if not lane_info.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right
    if len(polygon_coords) >= 3:
        poly = plt.Polygon(polygon_coords, closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
        ax.add_patch(poly)

# Lane Dividerを描画
for ld in lane_dividers:
    if len(ld.points) < 2:
        continue
    xs = [p[0] for p in ld.points]
    ys = [p[1] for p in ld.points]
    ax.plot(xs, ys, color='blue', linewidth=1.0, alpha=0.7)

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Lane Dividers ({len(lane_dividers)} lines)')
plt.tight_layout()
plt.show()

In [ ]:
# Crosswalkの可視化
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてレーンポリゴンを描画
for lane_info in lanes.values():
    if not lane_info.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right
    if len(polygon_coords) >= 3:
        poly = plt.Polygon(polygon_coords, closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
        ax.add_patch(poly)

# Crosswalkを描画
crosswalks = carla_map_data['crosswalks']
n_crosswalks = len(crosswalks)
cmap = plt.cm.get_cmap('Set2', max(n_crosswalks, 1))

for i, cw in enumerate(crosswalks):
    if len(cw.vertices) < 3:
        continue
    color = cmap(i % 8)
    poly = plt.Polygon(cw.vertices, closed=True, facecolor=color, edgecolor='black', linewidth=1.0, alpha=0.7)
    ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Crosswalks ({n_crosswalks} polygons)')
plt.tight_layout()
plt.show()

In [ ]:
# Sidewalkの可視化
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてレーンポリゴンを描画
for lane_info in lanes.values():
    if not lane_info.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right
    if len(polygon_coords) >= 3:
        poly = plt.Polygon(polygon_coords, closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
        ax.add_patch(poly)

# Sidewalkを描画
sidewalks = carla_map_data['sidewalks']
n_sidewalks = len(sidewalks)
cmap = plt.cm.get_cmap('Pastel1', max(n_sidewalks, 1))

for i, (key, sw) in enumerate(sidewalks.items()):
    if not sw.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in sw.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(sw.boundary_points)]
    polygon_coords = left + right
    if len(polygon_coords) < 3:
        continue
    color = cmap(i % 9)
    poly = plt.Polygon(polygon_coords, closed=True, facecolor=color, edgecolor='brown', linewidth=0.5, alpha=0.7)
    ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Sidewalks ({n_sidewalks} segments)')
plt.tight_layout()
plt.show()

In [ ]:
# Stop Lineの元データの可視化（nuScenesのstop_lineレイヤーの生成元となる抽出データをタイプ別に描画）
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてレーンポリゴンを描画
for lane_info in lanes.values():
    if not lane_info.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right
    if len(polygon_coords) >= 3:
        poly = plt.Polygon(polygon_coords, closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
        ax.add_patch(poly)

# 横断歩道由来の停止線（converterはcrosswalkの最初の一辺から生成する）
crosswalks = carla_map_data['crosswalks']
n_ped = 0
for cw in crosswalks:
    if len(cw.vertices) < 2:
        continue
    p1, p2 = cw.vertices[0], cw.vertices[1]
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='orange', linewidth=3.0, alpha=0.8, zorder=2)
    n_ped += 1

# 信号機由来の停止線（アクターの停止waypointから生成）
traffic_light_actors = carla_map_data['traffic_light_actors']
n_tl = 0
for tl in traffic_light_actors:
    if len(tl.stop_line_points) < 2:
        continue
    xs = [p[0] for p in tl.stop_line_points]
    ys = [p[1] for p in tl.stop_line_points]
    ax.plot(xs, ys, color='green', linewidth=3.0, alpha=0.8, zorder=2)
    n_tl += 1

# 停止標識の位置（現状converterではstop_line未生成。参考として位置のみ描画）
stop_signs = carla_map_data['stop_signs']
for ss in stop_signs:
    ax.plot(ss.x, ss.y, marker='^', markersize=10, color='red', markeredgecolor='black', zorder=3)

# 凡例
legend_handles = [
    Line2D([0], [0], color='orange', linewidth=3.0, label=f'PED_CROSSING ({n_ped})'),
    Line2D([0], [0], color='green', linewidth=3.0, label=f'TRAFFIC_LIGHT ({n_tl})'),
    Line2D([0], [0], color='red', marker='^', linestyle='None', markersize=10,
           markeredgecolor='black', label=f'STOP_SIGN ({len(stop_signs)})'),
]
ax.legend(handles=legend_handles, loc='upper right')

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Stop Line sources ({n_ped} ped crossings, {n_tl} traffic lights, {len(stop_signs)} stop signs)')
plt.tight_layout()
plt.show()

In [ ]:
# 信号機（アクター）の可視化
import math
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてレーンポリゴンを描画
for lane_info in lanes.values():
    if not lane_info.boundary_points:
        continue
    left = [(bp.left_x, bp.left_y) for bp in lane_info.boundary_points]
    right = [(bp.right_x, bp.right_y) for bp in reversed(lane_info.boundary_points)]
    polygon_coords = left + right
    if len(polygon_coords) >= 3:
        poly = plt.Polygon(polygon_coords, closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
        ax.add_patch(poly)

# 信号機を描画（ポール位置 + 正面方向矢印 + 灯器ヘッド + 停止線 + 対応線）
traffic_light_actors = carla_map_data['traffic_light_actors']
n_traffic_lights = len(traffic_light_actors)
cmap = plt.cm.get_cmap('tab10', 10)

ARROW_LENGTH = 5.0  # 正面方向矢印の長さ (m)

for i, tl in enumerate(traffic_light_actors):
    color = cmap(i % 10)
    # 灯器位置（信号機が取り付けられているポールの位置を示す）
    ax.plot(tl.x, tl.y, marker='o', markersize=8, color=color, zorder=3)
    # 灯器正面（ドライバーから見える面）方向の矢印（制御レーン進行方向の逆向き）
    yaw_rad = math.radians(tl.yaw)
    ax.annotate(
        '', xytext=(tl.x, tl.y),
        xy=(tl.x + ARROW_LENGTH * math.cos(yaw_rad), tl.y + ARROW_LENGTH * math.sin(yaw_rad)),
        arrowprops=dict(arrowstyle='->', color=color, linewidth=1.5), zorder=3,
    )
    # 灯器ヘッド（筐体）の中心位置
    for head in tl.light_heads:
        ax.plot(head.x, head.y, marker='x', markersize=6, color=color, zorder=3)
    # 停止線
    if len(tl.stop_line_points) >= 2:
        xs = [p[0] for p in tl.stop_line_points]
        ys = [p[1] for p in tl.stop_line_points]
        ax.plot(xs, ys, color=color, linewidth=3.0, alpha=0.8, zorder=2)
        # 灯器と停止線中点を結ぶ細い破線（対応関係を明示）
        mid_x = sum(xs) / len(xs)
        mid_y = sum(ys) / len(ys)
        ax.plot([tl.x, mid_x], [tl.y, mid_y], color=color, linewidth=0.8, linestyle='--', alpha=0.6, zorder=1)

ax.set_aspect('equal')
ax.autoscale_view()
ax.invert_yaxis()  # CARLAは左手系(Y軸南向き)のためY軸を下向きに表示
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Traffic Light Actors ({n_traffic_lights} lights, arrow=facing, x=light head, thick line=stop line)')
plt.tight_layout()
plt.show()